In [ ]:
%load_ext autoreload
%autoreload 2


from src.config import Configuration
import numpy as np
CONFIG = Configuration()

# Load ViT and Data

Set `MODEL_SOURCE = "google"` (HuggingFace) or `MODEL_SOURCE = "timm"` (PyTorch Image Models). Both produce `layer_attentions` of shape `(1, 12, 197, 197)` per layer — the rest of the notebook works identically.

1. **Zebra** (340) - High contrast stripes.
2. **Dalmatian** (251) - Distinct spot patterns.
3. **Monarch butterfly** (323) - Clear shape and contrast against nature backgrounds.
4. **Acoustic guitar** (401) - Strong geometric lines.
5. **School bus** (779) - Large, uniform color block.
6. **Volcano** (980) - Distinct peak structure.
7. **Daisy** (985) - Radial symmetry.
8. **Peacock** (84) - Complex, detailed texture.
9. **Brain coral** (109) - High-frequency texture testing.
10. **Espresso maker** (550) - Metallic, reflective, complex object.

In [ ]:
import os
from maikol_utils.file_utils import list_dir_files

all_images, _ = list_dir_files(CONFIG.attention_data, absolute_path=True)
classes = set([os.path.basename(img).split("_")[0] for img in all_images])

classes

In [ ]:
MODEL_SOURCE = "google" 
# MODEL_SOURCE = "timm" 

if MODEL_SOURCE == "google":
    from transformers import ViTModel
    model = ViTModel.from_pretrained(
        'google/vit-base-patch16-224', 
        output_attentions=True # NOTE
    )
elif MODEL_SOURCE == "timm":
    import timm
    model = timm.create_model('vit_base_patch16_224', pretrained=True)

model.eval()

# Visualization

### Get attention

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from types import SimpleNamespace

to_tensor = transforms.ToTensor()

def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((224, 224)) 
    image = to_tensor(image).unsqueeze(0)  # (1, 3, H, W)
    return image


if MODEL_SOURCE == "google":
    with torch.no_grad():
        outputs = {
            path: model(load_image(path)) for path in all_images
        }
    layer_attentions = outputs[all_images[0]].attentions
elif MODEL_SOURCE == "timm":
    _timm_attentions = []
    _timm_num_heads = 12
    _timm_head_dim = 64

    def _timm_attn_hook(module, input, output):
        qkv = output.detach().cpu()
        B, N, C = qkv.shape
        qkv = qkv.reshape(B, N, 3, _timm_num_heads, _timm_head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scale = _timm_head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        _timm_attentions.append(attn)

    _timm_handles = []
    for name, module in model.named_modules():
        if '.qkv' in name and 'attn.qkv' in name:
            _timm_handles.append(module.register_forward_hook(_timm_attn_hook))

    outputs = {}
    with torch.no_grad():
        for path in all_images:
            _timm_attentions.clear()
            model(load_image(path))
            outputs[path] = SimpleNamespace(attentions=[a for a in _timm_attentions])

    for h in _timm_handles:
        h.remove()

    layer_attentions = outputs[all_images[0]].attentions

print("Number of attention layers:", len(layer_attentions)) 
print("Shape of the the attention:", layer_attentions[1].shape) 
layer_attentions

### Layer CLS token attentio over all the paches at layer $l$

- We have 224×224 image separed into 14×14 paches: total of 196 paches.
- We had 197 bc it was $196 \text{(PATCHES)} + 1 \text{(CLS)}$

In [ ]:
from typing import Literal
from maikol_utils.print_utils import print_separator

def full_attention_at_layer(attention, layer_idx, type: Literal["mean", "max", "heads"] = "mean"):
    if type == "mean":
        return attention[layer_idx][0].mean(dim=0)
    elif type == "max":
        return attention[layer_idx][0].max(dim=0)[0]
    elif type == "heads":
        return attention[layer_idx][0]
    
def cls_attention_at_layer(attention, layer_idx, type: Literal["mean", "max", "heads"] = "mean"):
    if type == "mean":
        return attention[layer_idx][0].mean(dim=0)[0, 1:]
    elif type == "max":
        return attention[layer_idx][0].max(dim=0)[0][0, 1:]
    elif type == "heads":
        return attention[layer_idx][0][:, 0, 1:]
    
print_separator('Full attention at layer 0')
print(full_attention_at_layer(layer_attentions, 0, type="mean").shape)
print(full_attention_at_layer(layer_attentions, 0, type="max").shape)
print(full_attention_at_layer(layer_attentions, 0, type="heads").shape)

print_separator('CLS attention at layer 0')
print(cls_attention_at_layer(layer_attentions, 0, type="mean").shape)
print(cls_attention_at_layer(layer_attentions, 0, type="max").shape)
print(cls_attention_at_layer(layer_attentions, 0, type="heads").shape)

In [ ]:
import os
import importlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image


def _to_attention_grid(attn_1d):
    grid_size = int(np.sqrt(attn_1d.shape[-1]))
    return attn_1d.reshape(grid_size, grid_size)


def _resize_attention_map(attn_1d, target_size=(224, 224)):
    attn_grid = _to_attention_grid(attn_1d)
    return cv2.resize(attn_grid, target_size, interpolation=cv2.INTER_CUBIC)


def _normalize_map(attn_map):
    attn_min = float(attn_map.min())
    attn_max = float(attn_map.max())
    if attn_max - attn_min < 1e-8:
        return np.zeros_like(attn_map, dtype=np.float32)
    return ((attn_map - attn_min) / (attn_max - attn_min)).astype(np.float32)


def _extract_label_from_filename(image_path):
    # Approximate predicted label from filename prefix (e.g., "zebra_*.jpg" -> "zebra")
    return os.path.basename(image_path).split("_")[0]


def _global_cls_max(attention):
    """Global max CLS attention value across all layers and heads for consistent color scaling."""
    return float(max(layer[0][:, 0, 1:].max().item() for layer in attention))


def _build_single_map_figure(output, image_path, mode="mean", layer_idx=0, scale="global"):
    attention = output.attentions
    num_layers = len(attention)

    if layer_idx < 0 or layer_idx >= num_layers:
        raise ValueError(f"layer_idx must be in [0, {num_layers - 1}]")

    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))

    attn_1d = cls_attention_at_layer(attention, layer_idx, type=mode).detach().cpu().numpy()
    attn_map = _resize_attention_map(attn_1d, target_size=(224, 224))
    zmax_val = _global_cls_max(attention) if scale == "global" else 1.0
    if scale == "local":
        attn_map = _normalize_map(attn_map)

    fig = go.Figure()
    fig.add_trace(go.Image(z=img))
    fig.add_trace(go.Heatmap(
        z=attn_map,
        opacity=0.5,
        colorscale="Jet",
        showscale=False,
        zmin=0,
        zmax=zmax_val
    ))

    predicted_label = _extract_label_from_filename(image_path)

    fig.update_layout(
        title=f"ViT Attention ({mode.upper()}) - Layer {layer_idx}",
        margin=dict(l=40, r=40, t=40, b=40),
        annotations=[
            dict(
                x=0.01,
                y=0.99,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    return fig


def _build_mean_max_side_by_side_figure(output, image_path, layer_idx=0, scale="global"):
    attention = output.attentions
    num_layers = len(attention)

    if layer_idx < 0 or layer_idx >= num_layers:
        raise ValueError(f"layer_idx must be in [0, {num_layers - 1}]")

    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))

    mean_1d = cls_attention_at_layer(attention, layer_idx, type="mean").detach().cpu().numpy()
    max_1d = cls_attention_at_layer(attention, layer_idx, type="max").detach().cpu().numpy()

    mean_map = _resize_attention_map(mean_1d, target_size=(224, 224))
    max_map = _resize_attention_map(max_1d, target_size=(224, 224))
    zmax_val = _global_cls_max(attention) if scale == "global" else 1.0
    if scale == "local":
        mean_map = _normalize_map(mean_map)
        max_map = _normalize_map(max_map)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Attention (MEAN)", "Attention (MAX)"],
        horizontal_spacing=0.06
    )

    fig.add_trace(go.Image(z=img), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=mean_map,
        opacity=0.5,
        colorscale="Jet",
        showscale=False,
        zmin=0,
        zmax=zmax_val
    ), row=1, col=1)

    fig.add_trace(go.Image(z=img), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=max_map,
        opacity=0.5,
        colorscale="Jet",
        showscale=False,
        zmin=0,
        zmax=zmax_val
    ), row=1, col=2)

    predicted_label = _extract_label_from_filename(image_path)

    fig.update_layout(
        title=f"ViT Attention (Mean vs Max) - Layer {layer_idx}",
        margin=dict(l=40, r=40, t=60, b=40),
        annotations=list(fig.layout.annotations) + [
            dict(
                x=0.01,
                y=1.08,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    return fig


def _build_heads_subplot_figure(output, image_path, layer_idx=0, scale="global"):
    attention = output.attentions
    num_layers = len(attention)

    if layer_idx < 0 or layer_idx >= num_layers:
        raise ValueError(f"layer_idx must be in [0, {num_layers - 1}]")

    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))
    heads_attn = cls_attention_at_layer(attention, layer_idx, type="heads").detach().cpu().numpy()  # (num_heads, 196)
    global_max = _global_cls_max(attention)

    num_heads = heads_attn.shape[0]
    rows = 2
    cols = int(np.ceil(num_heads / rows))

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"Head {h}" for h in range(num_heads)],
        horizontal_spacing=0.02,
        vertical_spacing=0.10
    )

    for h in range(num_heads):
        row = h // cols + 1
        col = h % cols + 1

        attn_map = _resize_attention_map(heads_attn[h], target_size=(224, 224))
        if scale == "local":
            norm_map = _normalize_map(attn_map)
            heat_u8 = np.uint8(norm_map * 255.0)
            heat_rgb = cv2.cvtColor(cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        else:
            norm_map = np.clip(attn_map, 0.0, global_max) / max(global_max, 1e-8)
            heat_rgb = (plt.cm.jet(norm_map)[:, :, :3] * 255).astype(np.uint8)

        # Blend attention with the original image for each head panel.
        overlay = cv2.addWeighted(img, 0.55, heat_rgb, 0.45, 0)

        fig.add_trace(go.Image(z=overlay), row=row, col=col)

    predicted_label = _extract_label_from_filename(image_path)

    fig.update_layout(
        title=f"ViT Attention Heads (Layer {layer_idx})",
        margin=dict(l=20, r=20, t=60, b=20),
        showlegend=False,
        annotations=[
            dict(
                x=0.01,
                y=1.06,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    return fig




def visualize_attention(output, image_path, mode="combined", layer_idx=0):
    attention = output.attentions
    num_layers = len(attention)

    widgets_spec = importlib.util.find_spec("ipywidgets")

    # Backward compatibility for previous API values.
    if mode in {"mean", "max"}:
        mode = "combined"

    # Interactive controls when ipywidgets is available.
    if widgets_spec is not None:
        widgets = importlib.import_module("ipywidgets")
        ipy_display = importlib.import_module("IPython.display")

        mode_buttons = widgets.ToggleButtons(
            options=[("Mean + Max", "combined"), ("Heads", "heads")],
            value=mode if mode in {"combined", "heads"} else "combined",
            description="Mode:"
        )

        scale_buttons = widgets.ToggleButtons(
            options=[("Global", "global"), ("Local", "local")],
            value="global",
            description="Scale:"
        )

        layer_slider = widgets.IntSlider(
            value=max(0, min(layer_idx, num_layers - 1)),
            min=0,
            max=num_layers - 1,
            step=1,
            description="Layer:",
            continuous_update=False
        )

        out = widgets.Output()

        def _render(*_):
            with out:
                ipy_display.clear_output(wait=True)
                if mode_buttons.value == "heads":
                    fig = _build_heads_subplot_figure(
                        output=output,
                        image_path=image_path,
                        layer_idx=layer_slider.value,
                        scale=scale_buttons.value
                    )
                else:
                    fig = _build_mean_max_side_by_side_figure(
                        output=output,
                        image_path=image_path,
                        layer_idx=layer_slider.value,
                        scale=scale_buttons.value
                    )
                fig.show()

        mode_buttons.observe(_render, names="value")
        scale_buttons.observe(_render, names="value")
        layer_slider.observe(_render, names="value")

        controls = widgets.HBox([mode_buttons, scale_buttons, layer_slider])
        ipy_display.display(widgets.VBox([controls, out]))
        _render()
        return

    # Fallback without widgets: render one selected view.
    if mode == "heads":
        fig = _build_heads_subplot_figure(output=output, image_path=image_path, layer_idx=layer_idx)
    else:
        fig = _build_mean_max_side_by_side_figure(output=output, image_path=image_path, layer_idx=layer_idx)

    fig.show()
    print("ipywidgets not installed: pass mode='combined'|'heads' and layer_idx manually.")


def visualize_attention_interactive(all_images_list, outputs_dict):
    """Interactive per-layer attention visualization with image selector."""
    widgets_spec = importlib.util.find_spec("ipywidgets")

    if widgets_spec is None:
        print("ipywidgets not installed. Call visualize_attention(outputs[image_path], image_path) manually.")
        return

    widgets = importlib.import_module("ipywidgets")
    ipy_display = importlib.import_module("IPython.display")

    # Create image dropdown with basenames
    image_options = [(os.path.basename(img_path), img_path) for img_path in all_images_list]

    image_dropdown = widgets.Dropdown(
        options=image_options,
        value=all_images_list[0] if all_images_list else None,
        description="Image:"
    )

    # Get first output to determine num layers
    first_output = outputs_dict[all_images_list[0]]
    num_layers = len(first_output.attentions)

    mode_buttons = widgets.ToggleButtons(
        options=[("Mean + Max", "combined"), ("Heads", "heads")],
        value="combined",
        description="Mode:"
    )

    scale_buttons = widgets.ToggleButtons(
        options=[("Global", "global"), ("Local", "local")],
        value="global",
        description="Scale:"
    )

    layer_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=num_layers - 1,
        step=1,
        description="Layer:",
        continuous_update=False
    )

    out = widgets.Output()

    def _render(*_):
        with out:
            ipy_display.clear_output(wait=True)
            selected_image_path = image_dropdown.value
            output = outputs_dict[selected_image_path]

            if mode_buttons.value == "heads":
                fig = _build_heads_subplot_figure(
                    output=output,
                    image_path=selected_image_path,
                    layer_idx=layer_slider.value,
                    scale=scale_buttons.value
                )
            else:
                fig = _build_mean_max_side_by_side_figure(
                    output=output,
                    image_path=selected_image_path,
                    layer_idx=layer_slider.value,
                    scale=scale_buttons.value
                )
            fig.show()

    image_dropdown.observe(_render, names="value")
    mode_buttons.observe(_render, names="value")
    scale_buttons.observe(_render, names="value")
    layer_slider.observe(_render, names="value")

    controls = widgets.HBox([image_dropdown, mode_buttons, scale_buttons, layer_slider])
    ipy_display.display(widgets.VBox([controls, out]))
    _render()

In [ ]:
# Example: single image with interactive controls
i = 2
# visualize_attention(outputs[all_images[i]], all_images[i])

# Example: interactive image selector with per-layer visualization
visualize_attention_interactive(all_images, outputs)

# Attention Rollout

* **Attention rollout:**
    * Feed image into ViT
    * Attention layers $[A^l_h]$: layer $l$ has $h$ attention heads with size $(N+1)\times(N+1)$ ($N$ is the number of image patch tokens + 1 CLS token)
    * Aggregate (can be done with mean, max, min...) the attention over heads so that $[A^l_h]$ is reduced to $[A^l]$
    * Compute the attention rollout (estimating the flow of attention in the ViT network):
$$A_{rollout}^l = (A^l + I)A_{rollout}^{l-1}, \text{ for } i > 1$$
$$A_{rollout}^1 = A^1 + I$$

In [ ]:
def compute_attention_rollout(attentions, mode: Literal["mean", "max", "heads"] = "mean"):
    A = np.eye(197)  
    for layer in range(len(attentions)):
        attn = full_attention_at_layer(attentions, layer, type=mode).detach().numpy()
        attn = (attn + np.eye(197)) / attn.sum(axis=-1, keepdims=True)
        A = attn @ A

    rollout_cls = A[0, 1:]
    return rollout_cls


def _plot_rollout_single(output, image_path, scale="global"):
    """Internal function to render a single rollout visualization."""
    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))
    predicted_label = os.path.basename(image_path).split("_")[0]
    layer_attentions = output.attentions

    # Compute rollout for both modes
    rollout_cls_mean = compute_attention_rollout(layer_attentions, mode="mean")
    rollout_cls_max = compute_attention_rollout(layer_attentions, mode="max")

    # Reshape to 14x14 grids and resize to 224x224
    attn_grid_mean = rollout_cls_mean.reshape(14, 14)
    attn_grid_max = rollout_cls_max.reshape(14, 14)

    attn_map_mean = cv2.resize(attn_grid_mean, (224, 224), interpolation=cv2.INTER_CUBIC)
    attn_map_max = cv2.resize(attn_grid_max, (224, 224), interpolation=cv2.INTER_CUBIC)

    if scale == "local":
        attn_map_mean = _normalize_map(attn_map_mean)
        attn_map_max = _normalize_map(attn_map_max)
        zmax_mean = 1.0
        zmax_max = 1.0
    else:
        zmax_mean = float(attn_map_mean.max())
        zmax_max = float(attn_map_max.max())

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Attention Rollout (MEAN)", "Attention Rollout (MAX)"],
        horizontal_spacing=0.06
    )

    # Left subplot: mean
    fig.add_trace(go.Image(z=img), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=attn_map_mean,
        opacity=0.5,
        colorscale="Jet",
        showscale=False,
        zmin=0,
        zmax=zmax_mean
    ), row=1, col=1)

    # Right subplot: max
    fig.add_trace(go.Image(z=img), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=attn_map_max,
        opacity=0.5,
        colorscale="Jet",
        showscale=False,
        zmin=0,
        zmax=zmax_max
    ), row=1, col=2)

    fig.update_layout(
        title="Attention Rollout: Mean vs Max",
        margin=dict(l=40, r=40, t=60, b=40),
        annotations=list(fig.layout.annotations) + [
            dict(
                x=0.45,
                y=1.12,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig


def plot_rollout_attention(all_images_list, outputs_dict):
    """Interactive rollout visualization with image selector."""
    widgets_spec = importlib.util.find_spec("ipywidgets")
    
    if widgets_spec is None:
        print("ipywidgets not installed. Call _plot_rollout_single(outputs[image_path], image_path) manually.")
        return
    
    widgets = importlib.import_module("ipywidgets")
    ipy_display = importlib.import_module("IPython.display")
    
    # Create image dropdown with basenames
    image_options = [(os.path.basename(img_path), img_path) for img_path in all_images_list]
    
    image_dropdown = widgets.Dropdown(
        options=image_options,
        value=all_images_list[0] if all_images_list else None,
        description="Image:"
    )
    
    scale_buttons = widgets.ToggleButtons(
        options=[("Global", "global"), ("Local", "local")],
        value="global",
        description="Scale:"
    )
    
    out = widgets.Output()
    
    def _render(*_):
        with out:
            ipy_display.clear_output(wait=True)
            selected_image_path = image_dropdown.value
            fig = _plot_rollout_single(outputs_dict[selected_image_path], selected_image_path,
                                       scale=scale_buttons.value)
            fig.show()
    
    image_dropdown.observe(_render, names="value")
    scale_buttons.observe(_render, names="value")
    
    controls = widgets.HBox([image_dropdown, scale_buttons])
    ipy_display.display(widgets.VBox([controls, out]))
    _render()


# Example: interactive image selection with rollout visualization
plot_rollout_attention(all_images, outputs)